# 03 · Evolutionary Latent Clustering (ELC)
**Framework I of LPS — Clustering-based Persona Modeling**

Implements:
1. **HDBSCAN clustering** over user-prioritized embeddings → topical clusters $\{S_j\}$
2. **Time-decayed Cognitive Anchors**:

$$\mu_j = \frac{\sum_{t \in S_j} v_t \cdot e^{-\lambda(t_{\text{now}} - \tau_t)}}{\sum_{t \in S_j} e^{-\lambda(t_{\text{now}} - \tau_t)}}$$

3. **Query-to-anchor matching** via cosine similarity (Tier 1 retrieval)


In [ ]:

import sys
sys.path.append("../src")

import numpy as np
import pickle, yaml
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from clustering import ELCFramework

with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

with open("../data/embeddings.pkl", "rb") as f:
    all_embeddings = pickle.load(f)

uid = list(all_embeddings.keys())[0]
data = all_embeddings[uid]
print(f"User: {uid[:12]}... — {len(data['embeddings'])} interactions")


In [ ]:

# ── Fit ELC for one user ──────────────────────────────────────────────
elc = ELCFramework(
    lambda_decay=cfg["elc"]["lambda_decay"],
    min_cluster_size=cfg["elc"]["min_cluster_size"],
    min_samples=cfg["elc"]["min_samples"],
)

elc.fit(
    embeddings=data["embeddings"],
    timestamps=data["timestamps"],
)

print(f"Number of Cognitive Anchors (clusters): {len(elc.anchors_)}")
for anchor in elc.anchors_:
    print(f"  Cluster {anchor.cluster_id}: {len(anchor.member_indices)} members")


In [ ]:

# ── Visualise clusters in 2D PCA space ───────────────────────────────
pca = PCA(n_components=2)
pts = pca.fit_transform(data["embeddings"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw cluster labels
colors = plt.cm.tab10(np.linspace(0, 1, max(elc.labels_) + 2))
for label in set(elc.labels_):
    mask = elc.labels_ == label
    color = "lightgray" if label == -1 else colors[label]
    axes[0].scatter(pts[mask, 0], pts[mask, 1], c=[color], s=15, alpha=0.7,
                    label=f"Cluster {label}" if label != -1 else "Noise")
axes[0].set_title("HDBSCAN Clusters")
axes[0].legend(fontsize=7, loc="best")

# Right: cognitive anchor centroids (time-decayed)
axes[1].scatter(pts[:, 0], pts[:, 1], c="lightgray", s=10, alpha=0.4)
anchor_pts = pca.transform(np.stack([a.centroid for a in elc.anchors_]))
axes[1].scatter(anchor_pts[:, 0], anchor_pts[:, 1],
                c="red", s=120, marker="*", zorder=5, label="Cognitive Anchors μⱼ")
for i, (x, y) in enumerate(anchor_pts):
    axes[1].annotate(f"μ{i}", (x, y), fontsize=8, ha="left")
axes[1].set_title("Cognitive Anchors (Time-Decayed Centroids)")
axes[1].legend()

plt.suptitle(f"ELC Clustering — User {uid[:12]}...", fontsize=13)
plt.tight_layout()
plt.savefig("../data/elc_clusters.png", dpi=120)
plt.show()


In [ ]:

# ── Demonstrate temporal drift: compare early vs recent centroids ─────
import json
with open("../data/cohort.json") as f:
    cohort = json.load(f)

early_ts  = np.percentile(data["timestamps"], 25)
recent_ts = np.percentile(data["timestamps"], 75)

elc_early  = ELCFramework(**{k: cfg["elc"][k] for k in
                ["min_cluster_size","min_samples","lambda_decay"]})
elc_recent = ELCFramework(**{k: cfg["elc"][k] for k in
                ["min_cluster_size","min_samples","lambda_decay"]})

early_mask  = np.array(data["timestamps"]) <= early_ts
recent_mask = np.array(data["timestamps"]) >= recent_ts

if early_mask.sum() >= 5 and recent_mask.sum() >= 5:
    elc_early.fit(data["embeddings"][early_mask],
                  [t for t, m in zip(data["timestamps"], early_mask) if m])
    elc_recent.fit(data["embeddings"][recent_mask],
                   [t for t, m in zip(data["timestamps"], recent_mask) if m])
    print(f"Early  clusters : {len(elc_early.anchors_)}")
    print(f"Recent clusters : {len(elc_recent.anchors_)}")
    print("Topical drift observed: cluster centroids have shifted over time")
else:
    print("Not enough data for temporal drift analysis on this user sample")


In [ ]:

# ── Retrieve exemplars for a test query ──────────────────────────────
from embeddings import UserPrioritizedEmbedder

embedder = UserPrioritizedEmbedder(alpha=cfg["embedding"]["alpha"])
test_conv = list(cohort[uid]["test"])[0]
test_query = test_conv["pairs"][0]["user"]

query_emb = embedder.encode([test_query])[0]

exemplars = elc.retrieve_exemplars(
    query_embedding=query_emb,
    interactions=data["interactions"],
    top_k_anchors=cfg["retrieval"]["top_k_anchors"],
    top_k_exemplars=cfg["retrieval"]["top_k_exemplars"],
)

print(f"Query: {test_query[:100]}...")
print(f"\nRetrieved {len(exemplars)} exemplars:")
for i, ex in enumerate(exemplars, 1):
    print(f"  [{i}] {ex['user'][:80]}...")

print("\n✓ Notebook 03 complete — ELC framework ready")
